# Module 8B: DQ Dashboard - Python Visualization

## Learning Objectives
- Build rich DQ visualizations using matplotlib/plotly in Snowflake Notebooks
- Render an interactive dashboard without leaving the notebook

> **Business Value:** Data engineers get instant visual feedback without switching tools. Faster investigation = faster remediation.

---
> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes | **Variant:** Python (matplotlib/plotly)
> **Reference:** [STUDENT_GUIDE.md](../guide/STUDENT_GUIDE.md) — Module 8 compares the 3 dashboard variants (Native, Python, Streamlit).


> **What this does:** Sets your session context to the lab role, database, and warehouse.


In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE DQ_LAB_WH;

---
## Shared Setup: Create DQ Reporting Views

> **Note:** These views are identical across all Module 8 variants (8A/8B/8C). If you already ran another variant, these views already exist -- running them again is safe (`CREATE OR REPLACE`).

> **Business Value:** BI tools cannot call table functions directly. Views provide a stable, queryable interface.

In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_RESULTS_FLAT AS
SELECT
    r.REF_ENTITY_NAME AS TABLE_NAME, r.METRIC_NAME,
    r.ARGUMENT_NAMES AS COLUMN_CHECKED, r.VALUE AS METRIC_VALUE,
    r.EXPECTATION_NAME, r.EXPECTATION_RESULT, r.MEASUREMENT_TIME,
    COALESCE(c.SEVERITY, 'MEDIUM') AS SEVERITY,
    COALESCE(c.OWNER, 'Unassigned') AS RULE_OWNER,
    COALESCE(c.RULE_TYPE, 'SYSTEM') AS RULE_TYPE,
    CASE WHEN r.EXPECTATION_RESULT = 'MET' THEN 'PASS'
         WHEN r.EXPECTATION_RESULT = 'NOT_MET' THEN 'FAIL'
         ELSE 'NO_EXPECTATION' END AS STATUS
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'
)) r
LEFT JOIN CORP_DWH.DQ.RULES_CATALOG c
    ON UPPER(r.METRIC_NAME) LIKE '%' || REPLACE(UPPER(c.RULE_NAME), ' ', '_') || '%';

> **What this does:** Creates V_DQ_SCORECARD view that calculates per-table health scores from the latest DQ expectation results.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_SCORECARD AS
WITH latest AS (
    SELECT 'CORP_DWH.GOLD.DIM_CUSTOMER' AS TABLE_NAME,
        METRIC_NAME, ARGUMENT_NAMES, VALUE, EXPECTATION_NAME, EXPECTATION_RESULT, MEASUREMENT_TIME,
        ROW_NUMBER() OVER (PARTITION BY METRIC_NAME, ARGUMENT_NAMES ORDER BY MEASUREMENT_TIME DESC) AS RN
    FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
    WHERE EXPECTATION_NAME IS NOT NULL
)
SELECT TABLE_NAME,
    COUNT(*) AS TOTAL_EXPECTATIONS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS PASSED,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS FAILED,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS HEALTH_SCORE_PCT,
    MAX(MEASUREMENT_TIME) AS LAST_EVALUATED
FROM latest WHERE RN = 1 GROUP BY TABLE_NAME;

> **What this does:** Creates V_DQ_TREND view that provides hourly time-series data for charting quality metrics over time.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_TREND AS
SELECT DATE_TRUNC('HOUR', MEASUREMENT_TIME) AS MEASUREMENT_HOUR,
    METRIC_NAME, VALUE AS METRIC_VALUE, EXPECTATION_RESULT, MEASUREMENT_TIME
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL ORDER BY MEASUREMENT_TIME DESC;

> **What this does:** Creates V_DQ_EXECUTIVE_SUMMARY view that aggregates all checks into a single overall health percentage.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY AS
SELECT 'CORP_DWH' AS DATA_ESTATE, COUNT(*) AS TOTAL_CHECKS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS CHECKS_PASSING,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS CHECKS_FAILING,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS OVERALL_HEALTH_PCT,
    CURRENT_TIMESTAMP() AS AS_OF
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL;

---
## Dashboard: Executive Summary

> **What this does:** Renders an ASCII-art executive dashboard showing overall health score, pass/fail counts, and per-table health bars.


In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
session = get_active_session()

executive = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY").to_pandas()
scorecard = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_SCORECARD").to_pandas()

print("=" * 70)
print("            DATA QUALITY MONITORING DASHBOARD")
print("=" * 70)

if not executive.empty:
    health = float(executive['OVERALL_HEALTH_PCT'].iloc[0] or 0)
    total = int(executive['TOTAL_CHECKS'].iloc[0] or 0)
    passing = int(executive['CHECKS_PASSING'].iloc[0] or 0)
    failing = int(executive['CHECKS_FAILING'].iloc[0] or 0)
    bar_len = 50
    filled = int(bar_len * health / 100)
    bar = chr(9608) * filled + chr(9617) * (bar_len - filled)
    status = "HEALTHY" if health >= 90 else "DEGRADED" if health >= 70 else "CRITICAL"
    print(f"\n  Overall Health: {health:.0f}%  [{status}]")
    print(f"  [{bar}]")
    print(f"  Checks: {total} | Pass: {passing} | Fail: {failing}")
else:
    print("  [WAIT] No results yet.")

print("\n" + "-" * 70)
print("  HEALTH BY TABLE")
print("-" * 70)
if not scorecard.empty:
    for _, row in scorecard.iterrows():
        h = float(row['HEALTH_SCORE_PCT'] or 0)
        b = chr(9608) * int(20*h/100) + chr(9617) * (20 - int(20*h/100))
        print(f"  {row['TABLE_NAME']:<40} {h:>5.1f}% [{b}]")
print("=" * 70)

---
## Visual Charts (matplotlib)

> **What this does:** Generates matplotlib charts (pie, bar, gauge) for a visual DQ dashboard with pass/fail distribution, severity breakdown, and health score.


In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
session = get_active_session()

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    import numpy as np

    # Fetch all data
    results = session.sql(
        "SELECT METRIC_NAME, COLUMN_CHECKED, METRIC_VALUE, STATUS, SEVERITY, RULE_TYPE "
        "FROM CORP_DWH.DQ.V_DQ_RESULTS_FLAT WHERE STATUS IN ('PASS','FAIL')"
    ).to_pandas()
    
    exec_df = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY").to_pandas()
    
    trend_df = session.sql("""
        SELECT MEASUREMENT_HOUR,
            COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS PASSING,
            COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS FAILING
        FROM CORP_DWH.DQ.V_DQ_TREND
        GROUP BY MEASUREMENT_HOUR ORDER BY MEASUREMENT_HOUR
    """).to_pandas()

    scorecard = session.sql("SELECT * FROM CORP_DWH.DQ.V_DQ_SCORECARD").to_pandas()

    if results.empty:
        print("[INFO] No results to chart. Run DMFs with expectations first (Modules 1-4).")
    else:
        fig, axes = plt.subplots(3, 2, figsize=(16, 14))
        fig.suptitle('Data Quality Monitoring Dashboard', fontsize=16, fontweight='bold', y=0.98)
        plt.subplots_adjust(hspace=0.4, wspace=0.3)

        # 1. Health Gauge (top-left)
        ax = axes[0, 0]
        h = float(exec_df['OVERALL_HEALTH_PCT'].iloc[0] or 0) if not exec_df.empty else 0
        color = '#4CAF50' if h >= 90 else '#FF9800' if h >= 70 else '#f44336'
        ax.pie([h, 100-h], colors=[color, '#E0E0E0'], startangle=90,
               counterclock=False, wedgeprops=dict(width=0.35))
        ax.text(0, 0, f'{h:.0f}%', ha='center', va='center', fontsize=32, fontweight='bold', color=color)
        status_text = "HEALTHY" if h >= 90 else "DEGRADED" if h >= 70 else "CRITICAL"
        ax.text(0, -0.2, status_text, ha='center', va='center', fontsize=12, color=color)
        ax.set_title('Overall Health Score', fontsize=12, fontweight='bold')

        # 2. Pass/Fail Distribution (top-right)
        ax = axes[0, 1]
        counts = results['STATUS'].value_counts()
        colors_pf = {'PASS': '#4CAF50', 'FAIL': '#f44336'}
        bars = ax.bar(counts.index, counts.values, color=[colors_pf.get(x, '#999') for x in counts.index])
        ax.set_title('Pass/Fail Distribution', fontsize=12, fontweight='bold')
        ax.set_ylabel('Number of Checks')
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                    f'{int(bar.get_height())}', ha='center', va='bottom', fontweight='bold')

        # 3. Failures by Severity (middle-left)
        ax = axes[1, 0]
        fails = results[results['STATUS'] == 'FAIL']
        if not fails.empty:
            sev_order = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
            sev_counts = fails['SEVERITY'].value_counts()
            sev_colors = {'CRITICAL': '#D32F2F', 'HIGH': '#F44336', 'MEDIUM': '#FF9800', 'LOW': '#FFC107'}
            ordered = [(s, sev_counts.get(s, 0)) for s in sev_order if s in sev_counts]
            if ordered:
                labels, values = zip(*ordered)
                ax.barh(labels, values, color=[sev_colors.get(s, '#999') for s in labels])
                for i, v in enumerate(values):
                    ax.text(v + 0.1, i, str(v), va='center', fontweight='bold')
        ax.set_title('Failures by Severity', fontsize=12, fontweight='bold')
        ax.set_xlabel('Count')

        # 4. Top Violations (middle-right)
        ax = axes[1, 1]
        top = results[results['METRIC_VALUE'] > 0].nlargest(8, 'METRIC_VALUE')
        if not top.empty:
            names = [n[:22] for n in top['METRIC_NAME']]
            ax.barh(names, top['METRIC_VALUE'].values, color='#29B5E8')
            for i, v in enumerate(top['METRIC_VALUE'].values):
                ax.text(v + 0.1, i, str(int(v)), va='center', fontsize=9)
        ax.set_title('Top Violations (by count)', fontsize=12, fontweight='bold')
        ax.set_xlabel('Violation Count')
        ax.invert_yaxis()

        # 5. Quality Trend Over Time (bottom-left)
        ax = axes[2, 0]
        if not trend_df.empty and len(trend_df) > 1:
            ax.fill_between(range(len(trend_df)), trend_df['PASSING'], alpha=0.3, color='#4CAF50')
            ax.fill_between(range(len(trend_df)), trend_df['FAILING'], alpha=0.3, color='#f44336')
            ax.plot(range(len(trend_df)), trend_df['PASSING'], 'g-o', markersize=4, label='Passing')
            ax.plot(range(len(trend_df)), trend_df['FAILING'], 'r-o', markersize=4, label='Failing')
            ax.legend(loc='upper right')
            ax.set_xlabel('Measurement Period')
            ax.set_ylabel('Check Count')
        else:
            ax.text(0.5, 0.5, 'Insufficient trend data\n(need multiple DMF cycles)',
                    ha='center', va='center', transform=ax.transAxes, fontsize=11, color='#666')
        ax.set_title('Quality Trend', fontsize=12, fontweight='bold')

        # 6. Health by Table (bottom-right)
        ax = axes[2, 1]
        if not scorecard.empty:
            tables = [t.split('.')[-1] for t in scorecard['TABLE_NAME']]
            health_vals = scorecard['HEALTH_SCORE_PCT'].astype(float).values
            colors_h = ['#4CAF50' if h >= 90 else '#FF9800' if h >= 70 else '#f44336' for h in health_vals]
            bars = ax.barh(tables, health_vals, color=colors_h)
            ax.set_xlim(0, 105)
            for i, (bar, val) in enumerate(zip(bars, health_vals)):
                ax.text(val + 1, i, f'{val:.0f}%', va='center', fontweight='bold')
            ax.axvline(x=90, color='#4CAF50', linestyle='--', alpha=0.5, label='Healthy (90%)')
            ax.axvline(x=70, color='#FF9800', linestyle='--', alpha=0.5, label='Warning (70%)')
            ax.legend(loc='lower right', fontsize=8)
        else:
            ax.text(0.5, 0.5, 'No scorecard data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title('Health by Table', fontsize=12, fontweight='bold')
        ax.set_xlabel('Health %')

        plt.tight_layout()
        plt.savefig('/tmp/dq_dashboard.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("\n[OK] Dashboard rendered (3x2 grid: gauge, pass/fail, severity, violations, trend, per-table)")
        print("[OK] Also saved to /tmp/dq_dashboard.png")

except ImportError:
    print("[INFO] matplotlib not available in this notebook environment.")
    print("       Use the ASCII dashboard above, or try Module 8C (Streamlit).")


---
## Checkpoint

> **What this does:** Verifies your work so far. All checks should show [PASS].


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("=" * 50)
print("CHECKPOINT: Dashboard Views Ready")
print("=" * 50)
for v in ['V_DQ_RESULTS_FLAT', 'V_DQ_SCORECARD', 'V_DQ_TREND', 'V_DQ_EXECUTIVE_SUMMARY']:
    try:
        cnt = session.sql(f"SELECT COUNT(*) AS C FROM CORP_DWH.DQ.{v}").collect()[0]['C']
        print(f"  [PASS] {v} -- {cnt} rows")
    except Exception as e:
        print(f"  [FAIL] {v}: {str(e)[:50]}")
print("=" * 50)

---
**Next:** Try 8A (Native) or 8C (Streamlit), or proceed to `9_TEARDOWN`.